# Adversarial validation

Question: how different is the training domain from the test domain, which training data looks like test, and how much of the gap our dataset preparation closed?

A classifier learns to tell test texts from training texts. Out-of-fold ROC AUC near 0.5 means it cannot, so the domains look alike; AUC near 1.0 means they are easy to separate. Next to every BiGRU AUC we report the AUC of text length alone, to see how much of the gap is just length.

The assignment asks for adversarial validation with the same model family as the main task, so the classifier is a BiGRU over word embeddings. The main model is not chosen yet: `RNN` below is the one line to change once it is.

The notebook has two parts with the same procedure:

1. **Original data**: `uk_geo_dataset.csv` and `ru_geo_dataset.csv` as given, compared with test sentence by sentence.
2. **Normalised data**: `data/normalized/` built by `scripts/build_dataset.py`. Text goes through `src.text.prepare`, labels are fixed, sentences are glued into posts shaped like test. Test also goes through `prepare`, because that is what the model sees.

## 0. Setup

In [1]:
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

DATA = Path("../data")
PROJECT_ROOT = DATA.resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.text import prepare

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
RNN = nn.GRU
N_TRAIN, MAX_TOKENS, POST_TOKENS, EPOCHS = 30_000, 64, 256, 4
torch.manual_seed(0)
np.random.seed(0)
pd.set_option("display.max_colwidth", 160)

In [2]:
TOKEN = re.compile(r"\w+(?:['’ʼ-]\w+)*|[^\w\s]")
SENTENCE = re.compile(r"(?<=[.!?…])\s+|\n+")
strip_noise = lambda t: re.sub(r"[^\w\s'’ʼ-]|\d", " ", t)
strip_list = lambda xs: [strip_noise(x) for x in xs]
n_tokens = lambda xs: np.array([len(TOKEN.findall(x)) for x in xs])


class AdversarialRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.rnn = RNN(64, 64, batch_first=True, bidirectional=True)
        self.head = nn.Linear(128, 1)

    def forward(self, x):
        mask = (x > 0).float().unsqueeze(-1)
        output, _ = self.rnn(self.embedding(x))
        return self.head((output * mask).sum(1) / mask.sum(1).clamp(min=1)).squeeze(-1)


def adversarial_oof(positive, negative, epochs=EPOCHS, max_tokens=MAX_TOKENS):
    # Out-of-fold probability, 4 folds, that a text comes from `positive`.
    X = [[w.lower() for w in TOKEN.findall(x)][:max_tokens] or ["<empty>"] for x in positive + negative]
    y = np.r_[np.ones(len(positive)), np.zeros(len(negative))]
    oof = np.zeros(len(y))
    for fit_idx, hold_idx in StratifiedKFold(4, shuffle=True, random_state=0).split(X, y):
        counts = Counter(w for i in fit_idx for w in X[i])
        vocab = {w: i + 2 for i, (w, n) in enumerate(counts.most_common(30_000)) if n >= 2}
        encode = lambda idx: nn.utils.rnn.pad_sequence([torch.tensor([vocab.get(w, 1) for w in X[i]]) for i in idx], batch_first=True).to(DEVICE)
        model = AdversarialRNN(len(vocab) + 2).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), 2e-3)
        pos_weight = torch.tensor([float((y == 0).sum() / (y == 1).sum())], dtype=torch.float32, device=DEVICE)
        for _ in range(epochs):
            model.train()
            order = np.random.permutation(fit_idx)
            for start in range(0, len(order), 256):
                batch = order[start:start + 256]
                target = torch.tensor(y[batch], dtype=torch.float32, device=DEVICE)
                loss = nn.functional.binary_cross_entropy_with_logits(model(encode(batch)), target, pos_weight=pos_weight)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        model.eval()
        with torch.no_grad():
            for start in range(0, len(hold_idx), 1024):
                batch = hold_idx[start:start + 1024]
                oof[batch] = model(encode(batch)).cpu().numpy()
    return oof, y


def length_only_auc(positive, negative):
    auc = roc_auc_score(np.r_[np.ones(len(positive)), np.zeros(len(negative))], np.r_[n_tokens(positive), n_tokens(negative)])
    return max(auc, 1 - auc)


def auc_interval(oof, y, n=1000, seed=0):
    # 95% bootstrap interval, resampling positives and negatives separately.
    rng = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    aucs = []
    for _ in range(n):
        idx = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]
        aucs.append(roc_auc_score(y[idx], oof[idx]))
    return np.percentile(aucs, [2.5, 97.5])


RESULTS = {}


def compare(name, positive, negative, **kwargs):
    # Train the adversarial BiGRU, store the row for the final table, return (positive, negative) scores.
    oof, y = adversarial_oof(positive, negative, **kwargs)
    low, high = auc_interval(oof, y)
    RESULTS[name] = {"BiGRU AUC": round(roc_auc_score(y, oof), 3), "95% CI": f"{low:.3f}-{high:.3f}",
                     "length-only AUC": round(length_only_auc(positive, negative), 3),
                     "positives": len(positive), "negatives": len(negative)}
    return oof[:len(positive)], oof[len(positive):]


def show(*names):
    display(pd.DataFrame({name: RESULTS[name] for name in names}).T)

## 1. Original data

Train is sentence-level and test is post-level, so test posts are split into sentences of at least 15 characters; otherwise length alone would separate the domains. uk rows ending with the TCH section footer of one news site are dropped as a known artefact. ru is a 0.5% sample per chunk.

In [3]:
uk = pd.read_csv(DATA / "uk_geo_dataset.csv", usecols=["text", "is_valid"])
uk = uk[~uk["text"].str.contains(r"TCH\.?\s*$")]
test = pd.read_csv(DATA / "test.csv")
test_sents = [x.strip() for t in test["text"] for x in SENTENCE.split(t) if len(x.strip()) >= 15]

ru_parts = [c.sample(frac=0.005, random_state=i) for i, c in enumerate(pd.read_csv(DATA / "ru_geo_dataset.csv", usecols=["text"], chunksize=500_000))]
ru_sents = pd.concat(ru_parts)["text"].sample(N_TRAIN, random_state=0).tolist()
uk_train = uk.loc[uk["is_valid"].eq(0), "text"].sample(N_TRAIN, random_state=0).tolist()
uk_valid = uk.loc[uk["is_valid"].eq(1), "text"].tolist()
{"test sentences": len(test_sents), "uk train": len(uk_train), "uk is_valid": len(uk_valid), "ru": len(ru_sents)}

{'test sentences': 2384, 'uk train': 30000, 'uk is_valid': 9913, 'ru': 30000}

### 1.1 Can the BiGRU tell test sentences from uk train sentences?

In [4]:
compare("1.1 test vs uk train, sentences, raw", test_sents, uk_train)
test_scores_1, uk_scores_1 = compare("1.1 test vs uk train, sentences, no punctuation/emoji/digits", strip_list(test_sents), strip_list(uk_train))
show("1.1 test vs uk train, sentences, raw", "1.1 test vs uk train, sentences, no punctuation/emoji/digits")

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"1.1 test vs uk train, sentences, raw",0.87,0.861-0.878,0.677,2384,30000
"1.1 test vs uk train, sentences, no punctuation/emoji/digits",0.841,0.833-0.850,0.691,2384,30000


**Result.** AUC 0.870 on raw sentences and 0.841 without punctuation, emoji and digits; length alone gives 0.68-0.69 → train and test are different domains, and most of the gap is vocabulary and topics, not Telegram formatting.

### 1.2 How far is ru from test?

In [5]:
compare("1.2 test vs ru, sentences, no punctuation/emoji/digits", strip_list(test_sents), strip_list(ru_sents))
show("1.2 test vs ru, sentences, no punctuation/emoji/digits")

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"1.2 test vs ru, sentences, no punctuation/emoji/digits",0.988,0.986-0.991,0.726,2384,30000


**Result.** AUC 0.988 → ru is the most distant source, mostly because of language; test has only ~8% Russian posts, so ru is added in limited amounts.

### 1.3 Does uk `is_valid` look like uk train?

In [6]:
compare("1.3 uk is_valid vs uk train, sentences", strip_list(uk_valid), strip_list(uk_train), epochs=2)
show("1.3 uk is_valid vs uk train, sentences")

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"1.3 uk is_valid vs uk train, sentences",0.497,0.491-0.503,0.505,9913,30000


**Result.** AUC 0.497 → the given `is_valid` split is random within the train domain: it shows that a model learns, not how it will do on test.

### 1.4 Which uk train sentences look like test?

In [7]:
threshold_1 = np.median(test_scores_1)
print(f"uk train sentences scoring above the test median: {(uk_scores_1 > threshold_1).mean():.1%}")
display(pd.DataFrame({
    "test-like uk train": [uk_train[i] for i in np.argsort(-uk_scores_1)[:10]],
    "train-like test sentences": [test_sents[i] for i in np.argsort(test_scores_1)[:10]],
}))

uk train sentences scoring above the test median: 5.1%


,test-like uk train,train-like test sentences
0,"Режисер пояснив її так: “Це температура,.","Повний місяць також особливий тим, що це об’єднання Супермісяця та блакитного місяця (термін не має нічого спільного з кольором), пише Space."
1,"Встановлюються особи, причетні до вбивства"", - повідомили нам у поліції Київської області.",Також тут робив свої перші кроки в футболі єдиний в Києві олімпійський чемпіон з футболу 1988 року Олексій Михайличенко - ще зовсім малим він з батьками жив...
2,"Тож активісти для наших захисників ліплять вареники, печуть гречано-картопляні пироги та інші смаколики.","Це в 2,5 рази більше, ніж минулого року, коли лише 94 тисячі батьків вирішили відправити дітей на офлайн-навчання."
3,ДТП трапилася ввечері 2 вересня 2015 року.,"Щоб разом вчитись всьому новому: прогулянки з друзями, перші уроки, дивитись серіали у теплому ліжку та їсти смаколики в улюбленому кафе 🧡"", - йдеться в пов..."
4,Такий комплекс робіт проводять у 4—5 будинках кожного району.,Прийти та подивитися як він працює на нього краще у сонячну погоду.
5,"""5 канал"" транслює прес-конференція міністра освіти і науки Лілії Гриневич.","Перш за все, викликає питання сторінка, на якій опублікували оголошення, що отримало найбільшу кількість поширень."
6,У Києві на проспекті Академіка Заболотного вантажівка протаранила маршрутку.,Тепер вони використовують секрети лікування фахівців з передових країн.
7,"Люди стануть легковірними, прагнутимуть сенсацій.","Ціна, яку платить за своє вільне європейське майбутнє український народ, – надто висока."
8,"Основний товарний асортимент ХТЗ представлений малими тракторами потужністю 35 к/с, а також великими колісними і гусеничними тракторами потужністю 150-280 к/с.","Що означає, що цих свинособак дуже боляче покарали 💪 у різних областях їх недокраїни було дуже багато вибухів 👌"
9,Наразі територію пошкодження огороджено і спеціалісти КП «Київтеплоенерго» почали роботи з ліквідації аварії.,"Проєкт ""Створення рівних можливостей для сліпих і людей з вадами зору"" отримав високу міжнародну оцінку."


**Result.** Only 5.1% of uk train sentences score above the test median; the most test-like ones are Kyiv city news (accidents, police, municipal repairs) → a test-like slice exists but is small.

## 2. Normalised data

Sources come from `data/normalized/`: `uk_sentences.parquet` holds the cleaned uk sentences before gluing, `train.parquet` and `val.parquet` hold the posts used for training. Test goes through `prepare`, the same function the model input uses, so emoji, non-breaking spaces and channel signatures are already blanked.

Posts are longer than sentences: their p99 is about 300 tokens, so post comparisons read up to `POST_TOKENS = 256` tokens instead of 64. Test has only 477 posts, so intervals in 2.2-2.4 are wider than in part 1.

In [8]:
NORMALIZED = DATA / "normalized"
test_posts = [prepare(t) for t in test["text"]]
test_sents_norm = [x.strip() for t in test_posts for x in SENTENCE.split(t) if len(x.strip()) >= 15]

sentences = pd.read_parquet(NORMALIZED / "uk_sentences.parquet", columns=["text", "split"])
uk_train_norm = sentences.loc[sentences["split"].eq("train"), "text"].sample(N_TRAIN, random_state=0).tolist()
train_posts = pd.read_parquet(NORMALIZED / "train.parquet", columns=["text", "lang"])
val_posts = pd.read_parquet(NORMALIZED / "val.parquet", columns=["text", "lang"])
train_sample = train_posts["text"].sample(N_TRAIN, random_state=0).tolist()
by_lang = {lang: group["text"].sample(min(N_TRAIN, len(group)), random_state=0).tolist() for lang, group in train_posts.groupby("lang")}
{"test posts": len(test_posts), "test sentences": len(test_sents_norm), "uk train sentences": len(uk_train_norm),
 "train posts": len(train_sample), **{f"train posts, {lang}": len(x) for lang, x in by_lang.items()}, "val posts": len(val_posts)}

{'test posts': 477,
 'test sentences': 2204,
 'uk train sentences': 30000,
 'train posts': 30000,
 'train posts, ru': 30000,
 'train posts, uk': 30000,
 'val posts': 13265}

### 2.1 After cleaning and normalisation, can the BiGRU still tell test sentences from uk train sentences?

In [9]:
compare("2.1 test vs uk train, sentences, raw", test_sents_norm, uk_train_norm)
compare("2.1 test vs uk train, sentences, no punctuation/emoji/digits", strip_list(test_sents_norm), strip_list(uk_train_norm))
show("2.1 test vs uk train, sentences, raw", "2.1 test vs uk train, sentences, no punctuation/emoji/digits")

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"2.1 test vs uk train, sentences, raw",0.849,0.840-0.858,0.674,2204,30000
"2.1 test vs uk train, sentences, no punctuation/emoji/digits",0.828,0.819-0.838,0.671,2204,30000


**Result.** AUC 0.849 raw (0.870 before) and 0.828 without punctuation (0.841 before); the raw intervals barely touch → cleaning and `prepare` close a small, real part of the formatting gap, and the topic gap stays.

### 2.2 Can the BiGRU tell test posts from our train posts?

In [10]:
compare("2.2 test vs train, posts, raw", test_posts, train_sample, max_tokens=POST_TOKENS)
test_scores_2, train_scores_2 = compare("2.2 test vs train, posts, no punctuation/emoji/digits", strip_list(test_posts), strip_list(train_sample), max_tokens=POST_TOKENS)
show("2.2 test vs train, posts, raw", "2.2 test vs train, posts, no punctuation/emoji/digits")

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"2.2 test vs train, posts, raw",0.9,0.886-0.914,0.512,477,30000
"2.2 test vs train, posts, no punctuation/emoji/digits",0.837,0.819-0.856,0.512,477,30000


**Result.** Length-only AUC falls from 0.68 to 0.51: gluing removed the length signal. The BiGRU still separates posts with AUC 0.900 raw and 0.837 without punctuation and digits → our posts look like test by length, not by content; punctuation, digits and line layout still differ (0.90 vs 0.84).

### 2.3 Is ru still the most distant part of train?

In [11]:
for lang, posts in by_lang.items():
    compare(f"2.3 test vs train {lang}, posts", strip_list(test_posts), strip_list(posts), max_tokens=POST_TOKENS)
show(*[f"2.3 test vs train {lang}, posts" for lang in by_lang])

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"2.3 test vs train ru, posts",0.986,0.978-0.991,0.508,477,30000
"2.3 test vs train uk, posts",0.865,0.848-0.882,0.512,477,30000


**Result.** ru posts 0.986, uk posts 0.865 → gluing does not bring ru closer; ru stays a separate domain.

### 2.4 Is our val as far from test as train is?

In [12]:
compare("2.4 test vs val, posts", strip_list(test_posts), strip_list(val_posts["text"].tolist()), max_tokens=POST_TOKENS)
show("2.4 test vs val, posts", "2.2 test vs train, posts, no punctuation/emoji/digits")

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"2.4 test vs val, posts",0.796,0.775-0.817,0.51,477,13265
"2.2 test vs train, posts, no punctuation/emoji/digits",0.837,0.819-0.856,0.512,477,30000


**Result.** Test vs val 0.796 against test vs train 0.837. Val has 8% ru and train 20%, and ru is trivially separable, so most of the difference is the ru share → val is not a closer proxy of test.

### 2.5 Are val and train one domain?

In [13]:
compare("2.5 val vs train, posts", strip_list(val_posts["text"].tolist()), strip_list(train_sample), epochs=2, max_tokens=POST_TOKENS)
show("2.5 val vs train, posts")

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"2.5 val vs train, posts",0.547,0.541-0.553,0.503,13265,30000


**Result.** AUC 0.547 (0.541-0.553): a small but real shift. The 8% vs 20% ru share alone explains about this much → no sign of leakage; the shift is the planned difference in language mix.

### 2.6 Which train posts look like test?

In [14]:
threshold_2 = np.median(test_scores_2)
print(f"train posts scoring above the test median: {(train_scores_2 > threshold_2).mean():.1%} (sentences in 1.4: {(uk_scores_1 > threshold_1).mean():.1%})")
display(pd.DataFrame({
    "test-like train posts": [train_sample[i][:300] for i in np.argsort(-train_scores_2)[:8]],
    "train-like test posts": [test_posts[i][:300] for i in np.argsort(test_scores_2)[:8]],
}))

train posts scoring above the test median: 5.0% (sentences in 1.4: 5.1%)


,test-like train posts,train-like test posts
0,Законодательство РФ запрещает лицам с двойным гражданством занимать руководящие посты на государственной службе.,"В останню ніч серпня люди могли спостерігати за Суперблакитним місяцем.\n\nЦе – рідкісна подія. Повний місяць також особливий тим, що це об'єднання Суперміс..."
1,"""Сьогодні тут зібралась справжня родина, яка завжди була та буде поруч із студентським спортом в місті Києві. Ми вдячні нашим друзям та партнерам за підтрим...",22-річний доброволець із Великобританії загинув в Україні\n\nЗагибель Семюеля Ньюї у боях на сході України підтвердило Міністерство закордонних справ Велико...
2,"У Києві вночі виникла пожежа електрокабелів у підземному колекторі по вулиці Хрещатик, 7. Станом на 9.33 - загоряння ліквідовано.","А ви знали, що біля Києво-Могилянської академії є сонячний годинник? \nПрийти та подивитися як він працює на нього краще у сонячну погоду. Цей незвичайни..."
3,На будівництво сміттєпереробного заводу на Львівщині потрібно ще 2 млн грн?,Ваш вибір?\n\n
4,"Зловмиснику повідомлено про підозру за ч. 1 ст. 152 (зґвалтування), ч. 2 ст. 186 (грабіж) та ч. 2 ст. 187 (розбій) Кримінального кодексу України та обрано з...",Вимагали гроші з підприємців та підпалювали їхнє майно - перед судом постануть 6 учасників злочинної організації\n \nЗа процесуального керівництва Київськ...
5,"Представники КП «Електроавтотранс» вже обстежили маршрут та задокументували зауваження-застереження по схемі руху, зупинках, тощо.","Нью-Йорку було дозволено стягувати плату з водіїв, які в'їжджають у Нижній Манхеттен, оголосила в понеділок губернатор Нью-Йорка Кеті Хочул. Про це повідомл..."
6,"ДТП сталася на проспекті Перемоги, 107 у напрямку центру столиці та заблокувала дві смуги руху.","Понад 5 тисяч безробітних доєдналися до відбудови Київщини у рамках державної програми ""Армія відновлення"" \n\nРегіон входить до лідерів за кількістю безр..."
7,"Леонід Швець: Реформатор має провести каденцію так, щоб пам'ятали.","Коломойський попросив не знімати його під час відкритої частини засідання\n\nЗагалом же засідання суду у справі Коломойського буде закритим, про це клопо..."


**Result.** 5.0% of train posts score above the test median, the same as 5.1% for sentences → the share of test-like material did not grow; the most test-like posts are again Kyiv city news, streets with house numbers and municipal services.

## 3. Conclusions

In [15]:
pd.DataFrame(RESULTS).T

,BiGRU AUC,95% CI,length-only AUC,positives,negatives
"1.1 test vs uk train, sentences, raw",0.87,0.861-0.878,0.677,2384,30000
"1.1 test vs uk train, sentences, no punctuation/emoji/digits",0.841,0.833-0.850,0.691,2384,30000
"1.2 test vs ru, sentences, no punctuation/emoji/digits",0.988,0.986-0.991,0.726,2384,30000
"1.3 uk is_valid vs uk train, sentences",0.497,0.491-0.503,0.505,9913,30000
"2.1 test vs uk train, sentences, raw",0.849,0.840-0.858,0.674,2204,30000
"2.1 test vs uk train, sentences, no punctuation/emoji/digits",0.828,0.819-0.838,0.671,2204,30000
"2.2 test vs train, posts, raw",0.9,0.886-0.914,0.512,477,30000
"2.2 test vs train, posts, no punctuation/emoji/digits",0.837,0.819-0.856,0.512,477,30000
"2.3 test vs train ru, posts",0.986,0.978-0.991,0.508,477,30000
"2.3 test vs train uk, posts",0.865,0.848-0.882,0.512,477,30000


| Comparison | Original (§1) | Normalised (§2) | Length-only, original → normalised |
|---|---|---|---|
| test vs uk train, sentences, raw | 0.870 | 0.849 | 0.68 → 0.67 |
| test vs uk train, sentences, no punctuation/digits | 0.841 | 0.828 | 0.69 → 0.67 |
| test vs train, posts, raw | — | 0.900 | 0.51 |
| test vs train, posts, no punctuation/digits | — | 0.837 | 0.51 |
| test vs ru | 0.988 | 0.986 | 0.73 → 0.51 |
| validation vs train | 0.497 (`is_valid`) | 0.547 (our val) | 0.51 → 0.50 |

- **Gluing closed the length gap, not the domain gap.** Length alone separated test from train with AUC 0.68; on our posts it gives 0.51. The BiGRU still separates test posts with AUC 0.84 without punctuation and digits, about as well as before.
- **Normalisation helps a little.** On sentences AUC drops by 0.02 (0.870 → 0.849 raw, 0.841 → 0.828 stripped). Emoji, apostrophes and spaces were part of the gap, but a small part.
- **What remains is content.** Test is local Kyiv Telegram news; train is national news with machine labels. The share of test-like training material stays at 5%. Formatting of glued posts also still differs: raw posts give 0.90 against 0.84 stripped.
- **ru stays a separate domain** (0.986), so keep its share limited.
- **Our val is a train-domain check, not a test proxy.** Val vs train 0.547 is explained by the 8% vs 20% ru share; test vs val 0.80. Use CV and val to see that the model learns, the labelled sample as a sanity check, and submissions for decisions that depend on the domain.
- **Noise.** Intervals are bootstraps over predictions and do not include retraining noise; with 477 test posts, differences below ~0.03 between post-level rows (e.g. 2.2 vs 2.3 uk) are not meaningful.

**What to try next.** The gap is in topics and vocabulary, so data formatting is mostly done. Pretrained fastText embeddings and more Kyiv and street material (substitution from a street list) target what is left.

**When the main model is chosen,** change `RNN` at the top and re-run, so that the check uses the same model as the task requires.